# SeaBASS data to CMAP

#### Krista Longnecker, 10 June 2026
BIOS-SCOPE has some data at SeaBASS that can be prepared for CMAP. SeaBASS data download is an offline process, so use data requested for download and stored locally (e.g., not synced to GitHub).

In [569]:
%reset -f

In [570]:
import os
import pandas as pd
from datetime import date

#SB_support from NASA's website: https://seabass.gsfc.nasa.gov/wiki/readsb_python
from SB_support import readSB


In [571]:
#Check that an Excel file exists for the variable metadata:
if os.path.exists('CMAP_variableMetadata_additions.xlsx'):
    print('Found Excel file with metadata')
else:
    #You cannot proceed without the file, so stop the script if it's not found
    print(f"No Excel file with metadata found")
    sys.exit(1)
    
#will add a new worksheet with the variables from Amy's data

Found Excel file with metadata


In [572]:
#go find the data in the data folder (it is in .gitignore, so it will not end up at GitHub). Makes a list
# I know the folder is there, so no need to make the folder
folder = "data"
SBstring = ".sb"
matching_files = []

os.chdir(".")

for root, dirs, files in os.walk(folder):
    for file in files:
        full_path = os.path.join(root, file)
        # Change 'file' to 'full_path' if you want to search the entire folder path
        if SBstring in file:  
            matching_files.append(full_path)

In [573]:
#get the list of cruises from the folder structure (needs this at the end)

# Target directory path
folder_path = 'data/SeaBASS/Maas/PVST_BATS_PLANKTON'

# Filter for folders only (requires joining path to verify accurately)
unCruises = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, f))]

In [574]:
#first up, read in the SeaBASS file and make a square matrix out of all the data sets. 
#I am going to group all cruises into one matrix

#read in the first file and use that to start the dataframe 
#otherwise I end up with rows grouped by cruise and I am not sure how to get around that
idx = 0
one = readSB(filename = matching_files[idx])
# #use the SeaBASS experiment name that will be used for the Excel file name
# dataset_short_name = one.headers['experiment'] 

#then process the data needed to start the data frame
oneD = one.data
df = pd.DataFrame(oneD)

#add a columm with filename while I sort out the issues
df['filename'] = one.filename

#then go through and add the remaining files...start loop at 2
for idx in range(2,(len(matching_files))):
    one = readSB(filename = matching_files[idx],no_warn=True)
    oneD = one.data
    temp = pd.DataFrame(oneD)
    temp['filename'] = one.filename

    if 'associatedmedia' in one.variables: # data file with details on zoop
        #print('work on this')

        #remove URL from associatedmedia and leave the object details (otherwise have issues in export)
        temp['associatedmedia'] = temp['associatedmedia'].str.strip('https://ecotaxa.obs-vlfr.fr/objectdetails/')
        temp = temp.rename(columns = {'associatedmedia': "objectdetails"})

        #get lat, lon
        temp['lat'] = (one.headers['north_latitude']).replace('[DEG]','') # was: 32.190[DEG]
        temp['lon'] = (one.headers['east_longitude']).replace('[DEG]','') # was: -64.532[DEG]  

        #then time
        time_str = one.headers['start_date'] + " " + (one.headers['start_time']).replace('[GMT]','')
        temp['ts'] = pd.to_datetime(time_str,format='%Y%m%d %H:%M:%S')             
        #now that I have added what is required, pop out of the if statement as the next steps are the same for both
    else:
        #print('normal datafile')
        #may as well convert time here as well (already have lat / lon)
        temp['ts'] = pd.to_datetime(temp['date'].astype(str) + ' ' + temp['time'].astype(str))
        #now that I have added what is required, pop out of the if statement as the next steps are the same for both

    #temp['date_cmap'] = temp['ts'].dt.strftime("%Y-%m-%dT%H:%M:%S" + "+00:00Z")
    temp['date_cmap'] = temp['ts'].dt.strftime("%Y-%m-%dT%H:%M:%S" + "Z")
    #now delete the temporary column 
    temp = temp.drop(columns = 'ts')                      
    df = pd.concat([df,temp],ignore_index=True)

In [575]:
#now get ready to put this into CMAP format
# Required variables are time, lat, lon, depth #df = pd.DataFrame(columns=['time','lat','lon','depth'])

target_col = "depth"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "lon"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "lat"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "date_cmap"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]


df = df.rename(columns = {'date_cmap': "time"})

In [576]:
df.to_excel('test.xlsx')

In [561]:
'''
Work on the second sheet: metadata about the variables; use the CMAP dataset template to setup the dataframe 
so I get the column headers right
'''
fName = 'datasetTemplate.xlsx' #downloaded from CMAP website
sheet_name = 'vars_meta_data'
vars = pd.read_excel(fName, sheet_name=sheet_name)
cols = vars.columns.tolist()
#df2 will be the dataframe with the metadata about the variables, set it up empty here
nVariables = df.shape[1]
df2 = pd.DataFrame(columns=cols,index = pd.RangeIndex(0,nVariables,1))

# this is only a partial list of variables for the moment
df2['var_short_name'] = df.columns
df2.loc[:,'var_long_name'] = ''
df2.loc[:,'var_sensor'] = ''
df2.loc[:,'var_unit'] = ''
df2.loc[:,('var_spatial_res')] = 'irregular'
df2.loc[:, ('var_temporal_res')] = 'irregular'
df2.loc[:,('var_discipline')] = 'biology'
df2.loc[:,('visualize')] = 0 #many would be hard to visualize these one at a time: edit later

#add the keywords, nothing fancy here
misc_keywords = 'BIOS-SCOPE, BIOS, Simons Foundation International, bio, biogeo, biogeochemistry, biology, flowCAM, Zooscan, net tows, Discrete, Bottle, cruise, Discrete, in situ, insitu, in-situ, North Atlantic Ocean, observation'
df2.loc[:,('var_keywords')] = misc_keywords
                         

In [562]:
#uncomment this out the first time to get the variable information, but then once the information is in Excel, read 
#the information that has been copied into fName = 'CMAP_variableMetadata_additions.xlsx'
#there is most certainly a better way to do this, but I am not going to deal with that now

# #SeaBASS does not appear to have a downloadable file with their variable names, though the information is here:
# #https://seabass.gsfc.nasa.gov/wiki/stdfields
# #copy/paste into Excel, tidy up, and use to get the details I need on units and long names
# fName = 'SeaBASS_variableNames.xlsx'
# vn = pd.read_excel(fName,sheet_name = 'variables')
# vn['var_short_name'] = vn['Field name'].str.lower() #otherwise I miss some of the matches :(

# df2 = pd.merge(df2,vn,how='left')
# df2['var_long_name'] = df2['Description']
# df2['var_unit'] = df2['Units']
# df2 = df2.drop(columns = ['Description','Units','Field name'])

In [ ]:
#there are a few pieces of metadata that CMAP wants that will be easier to track in an Excel file -
#make the file once, and then update as needed for future BCO-DMO datasets.
#The keywords include cruises, and all possible names for a variable. I wonder if
#CMAP has that information available in a way that can be searched?
#These are all one-offs...so it's rather hard to automate. I am coming to the idea it will be easier to manually annotate the files.
#Come back to this later.

# # Note that I made the Excel file after I started down this rabbit hole with the sensors. It will probably make sense
# #to pull the sensor information from the file as well.
fName = 'CMAP_variableMetadata_additions.xlsx'
sheetName = exportFile[0:31] #Excel limits the length of the sheet name
moreMD = pd.read_excel(fName,sheet_name = sheetName)

# #suffixes are added to column name to keep them separate; '' adds nothing while '_td' adds _td that can get deleted next
df2 = moreMD.merge(df2[['var_short_name','var_keywords']],on='var_short_name',how='right',suffixes=('', '_td',))
# # Discard the columns that acquired a suffix:
df2 = df2[[c for c in df2.columns if not c.endswith('_td')]]

In [564]:
'''
Gather the details I need about the dataset
'''
project_description = 'The data provided by this project is the abundance and taxonomic information for the 5-300 µm planktonic community captured via imaging with a FlowCam or Zooscan. We additionally analyze a duplicate subsetÂ of water samples from the same cast as BATS HPLC depths (0-250 m), sending these pigment samples to GSFC. Target depths for both the niskin-based FlowCam and HPLC samples include surface, 40, 80, 120, 160, and 200. The FlowCam was run in both trigger and automated mode at 10X for each bottle sample to enumerate and classify photosynthetic and non-photosynthetic organisms. In addition, the standard Bermuda Atlantic Time Series Site phytoplankton net tows (0-175 mwo, 120-150 m depth, 35 µm mesh net) were imaged. Images were taken using automatic mode at 2x, 4x, and 10x with the FlowCam, while two size fractions (>1000, <1000) were captured with a Zooscan to ensure enumeration of the larger phytoplankton taxonomic groups.'

# gather up the dataset_meta_data into df3

df3 = pd.DataFrame({
    'dataset_short_name': ['BS_PLANKTON_1'],
    'dataset_long_name': ['BS_PVST_BATS_PLANKTON'], 
    'dataset_version': ['1.0'],
    'dataset_release_date': [date.today()],
    'dataset_make': ['observation'],
    'dataset_source': ['Amy Maas, Bermuda Institute of Ocean Sciences - Arizona State University'],
    'dataset_distributor': ['Amy Maas, Bermuda Institute of Ocean Sciences - Arizona State University'],
    'dataset_acknowledgement': ['This work supported by funding from NASA.'],
    'dataset_history': [''],
    'dataset_description': [project_description],
    'dataset_references': ['DOI is 10.5067/SeaBASS/PVST_BATS_PLANKTON/DATA001; '],
    'climatology': [0]
    })

In [565]:
#insert the list of unique cruises from the data pulled from the folder structure
df3 = pd.concat([df3,pd.DataFrame(unCruises)],axis=1)
df3 = df3.rename(columns={df3.columns[-1]: 'cruise_names'})

In [566]:
fName_CMAP = 'data/' + 'BIOS-SCOPE_SeaBASS_' + 'PVST_BATS_PLANKTON' + '.xlsx'
dataset_names = {'data': df, 'dataset_meta_data': df3, 'vars_meta_data': df2}
with pd.ExcelWriter(fName_CMAP) as writer:
    for sheet_name, data in dataset_names.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)

In [567]:
# uncomment out the next line if I want to put code that will not be executed below this point
raise SystemExit("Stop execution here")

SystemExit: Stop execution here

C:\Users\LaneyLab\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3465: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
data.headers

In [ ]:
data.length

In [ ]:
data.bdl

In [ ]:
data.filename

In [ ]:
data.pi

In [ ]:
data.headers

In [ ]:
data.variables

In [ ]:
#then go through and add the remaining files...start loop at 2

for idx in range(2,(len(matching_files))):
    one = readSB(filename = matching_files[idx], no_warn=True)
    oneD = one.data
    temp = pd.DataFrame(oneD)
    d = [col for col in temp.columns if 'volume' in col]
    if d:
        print(one.filename)
    temp['filename'] = one.filename
    df_working = pd.concat([df_working,temp],ignore_index=True)

In [ ]:
# need to trap the files with the image information because they do not ahve the headers with the station information
# Add that in (and remove the URLs as they cannot be exported as an Excel file); will pull the objid from the URL
# in order to retain the information

idx = 4
one = readSB(filename = matching_files[idx],no_warn=True)
oneD = one.data
temp = pd.DataFrame(oneD)
temp['filename'] = one.filename

if 'associatedmedia' in one.variables: # data file with details on zoop
    #print('work on this')
    
    #remove URL from associatedmedia and leave the object details (otherwise have issues in export)
    temp['associatedmedia'] = temp['associatedmedia'].str.strip('https://ecotaxa.obs-vlfr.fr/objectdetails/')
    temp = temp.rename(columns = {'associatedmedia': "objectdetails"})
        
    #get lat, lon
    temp['lat'] = (one.headers['north_latitude']).replace('[DEG]','') # was: 32.190[DEG]
    temp['lon'] = (one.headers['east_longitude']).replace('[DEG]','') # was: -64.532[DEG]  
    
    #then time
    time_str = one.headers['start_date'] + " " + (one.headers['start_time']).replace('[GMT]','')
    temp['ts'] = pd.to_datetime(time_str,format='%Y%m%d %H:%M:%S')             
    #now that I have added what is required, pop out of the if statement as the next steps are the same for both
else:
    #print('normal datafile')
    #may as well convert time here as well (already have lat / lon)
    temp['ts'] = pd.to_datetime(temp['date'].astype(str) + ' ' + temp['time'].astype(str))
    #now that I have added what is required, pop out of the if statement as the next steps are the same for both
                   
#temp['date_cmap'] = temp['ts'].dt.strftime("%Y-%m-%dT%H:%M:%S" + "+00:00Z")
temp['date_cmap'] = temp['ts'].dt.strftime("%Y-%m-%dT%H:%M:%S" + "Z")
#now delete the temporary column 
temp = temp.drop(columns = 'ts')                      
df_working = pd.concat([df_working,temp],ignore_index=True)

In [ ]:
#then go through and add the remaining files...start loop at 2
for idx in range((len(matching_files))):
    one = readSB(filename = matching_files[idx], no_warn=True)
    oneD = one.data
    df_working = pd.DataFrame(oneD)
    d = [col for col in dfW.columns if 'volume' in col]
    if d:
        print(one.filename)